# NeuroWorkflow: Ring Network — 3 Point Neurons

Three `iaf_psc_alpha` neurons connected in a ring topology:
**popA → popB → popC → popA**

Demonstrates the **fan-in** `NW_Connectivity` node: all three populations
connect to a single connectivity node that defines the full connection matrix.

```
PopA ─┐
PopB ─┼─→ NW_Connectivity (ring matrix) → NW_SimConfig → NW_Analysis
PopC ─┘
```

Each neuron is driven by a constant bias current (`I_e`) so it fires
spontaneously. The ring connections modulate timing and phase relationships.

In [ ]:
from neuroworkflow import WorkflowBuilder
from neuroworkflow.nodes.network.NW_Population   import NW_Population
from neuroworkflow.nodes.network.NW_Connectivity import NW_Connectivity
from neuroworkflow.nodes.simulation.NW_SimConfig import NW_SimConfig
from neuroworkflow.nodes.analysis.NW_Analysis    import NW_Analysis

## 1. Instantiate nodes

In [ ]:
popA = NW_Population("PopA")
popB = NW_Population("PopB")
popC = NW_Population("PopC")

conn = NW_Connectivity("Connectivity")
cfg  = NW_SimConfig("SimConfig")
ana  = NW_Analysis("Analysis")

print(conn.get_info())

## 2. Configure nodes

Each neuron has a bias current `I_e = 400 pA` (just above rheobase for
`iaf_psc_alpha` with default parameters) so it fires spontaneously at ~5 Hz.
The ring connections couple the neurons and shift their relative timing.

In [ ]:
neuron_params = dict(
    model_type     = "point_neuron",
    model_template = "nest:iaf_psc_alpha",
    ei_type        = "exc",
    location       = "VISp",
    layer          = "L4",
    N              = 1,
    nest_params = {
        "C_m":     250.0,   # pF
        "tau_m":   10.0,    # ms
        "t_ref":   2.0,     # ms
        "V_th":    -55.0,   # mV
        "V_reset": -70.0,   # mV
        "E_L":     -70.0,   # mV
        "I_e":     400.0,   # pA  — bias current, drives spontaneous firing
    },
)

popA.configure(pop_name="popA", **neuron_params)
popB.configure(pop_name="popB", **neuron_params)
popC.configure(pop_name="popC", **neuron_params)

# Ring connectivity matrix: A→B→C→A
# iaf_psc_alpha is current-based → weights in pA.
# Demonstrates mixed synapse models per connection:
#   popA→popB : static_synapse  (fixed weight, no plasticity)
#   popB→popC : stdp_synapse    (weight adapts to spike timing)
#   popC→popA : static_synapse  (fixed weight, shorter delay)
conn.configure(
    connection_rule      = 1,
    model_template       = "static_synapse",
    dynamics_params      = "static_syn.json",
    dynamics_params_dict = {},             # empty — static_synapse needs no extra params
    delay                = 2.0,            # ms
    syn_weight           = 50.0,           # pA

    connections = [
        # A→B: static synapse, inherits node-level defaults
        {
            "source": "popA",
            "target": "popB",
        },
        # B→C: STDP synapse — weight evolves with pre/post spike timing
        {
            "source":               "popB",
            "target":               "popC",
            "model_template":       "stdp_synapse",
            "dynamics_params":      "stdp_syn.json",
            "dynamics_params_dict": {
                "tau_plus": 20.0,   # ms  — pre-before-post time constant (LTP window)
                "lambda":   0.01,   # learning rate
                "alpha":    1.0,    # ratio of LTD to LTP amplitude
                "mu_plus":  1.0,    # weight dependence exponent (LTP)
                "mu_minus": 1.0,    # weight dependence exponent (LTD)
                "Wmax":     500.0,  # pA  — maximum allowed weight
            },
        },
        # C→A: static synapse, faster feedback
        {
            "source": "popC",
            "target": "popA",
            "delay":  1.0,                 # ms  — shorter delay for this connection
        },
    ],
)

cfg.configure(
    simulator   = "pointnet",
    config_file = "config_ring.json",
    tstop_ms    = 3000.0,
    dt_ms       = 0.1,
    reports = {
        "v_report": {
            "variable_name": "V_m",
            "cells":         "all",
            "module":        "membrane_report",
            "sections":      "soma",
        }
    },
)

ana.configure(
    plot_raster  = True,
    plot_traces  = True,
    report_name  = "v_report",
    save_figures = True,
)

## 3. Build and run the workflow

All three populations connect to the **same** `Connectivity` fan-in port.
NeuroWorkflow collects them into a list; `NW_Connectivity` converts to a
`pop_name` lookup and applies the connection matrix.

In [ ]:
wf = WorkflowBuilder("NW_Ring_3Pop")

for node in [popA, popB, popC, conn, cfg, ana]:
    wf.add_node(node)

# Fan-in: all 3 populations → single Connectivity port
wf.connect("PopA", "population",  "Connectivity", "populations")
wf.connect("PopB", "population",  "Connectivity", "populations")
wf.connect("PopC", "population",  "Connectivity", "populations")

wf.connect("Connectivity", "network", "SimConfig", "populations")
wf.connect("SimConfig",    "results", "Analysis",  "results")

wf.context["results_path"] = "./results/ring"

workflow = wf.build()
ok = workflow.execute()
assert ok, "Workflow failed — check printed errors above"
print("Workflow completed successfully")

## 4. Validate outputs

In [ ]:
print("Output validation:")
for node_name, node in workflow.nodes.items():
    for port_name, port in node._output_ports.items():
        status = "OK" if port.value is not None else "*** None — node may have failed ***"
        print(f"  {node_name}.{port_name}: {status}")

## 5. Re-run with stronger coupling

Increase `syn_weight` to see tighter phase-locking between neurons.
No need to rebuild the workflow — just reconfigure and re-execute.

In [ ]:
conn.configure(syn_weight=50.0)
cfg.configure(config_file="config_ring_strong.json")

for node in workflow.nodes.values():
    node._context["results_path"] = "./results/ring_strong"

ok = workflow.execute()
assert ok, "Re-run failed"
print("Re-run complete")